In [1]:
import numpy as np
from dmipy.core.modeling_framework import MultiCompartmentModel
from dmipy.signal_models import sphere_models, cylinder_models, gaussian_models
from microtool.tissue_model import RelaxationTissueModel

from microtool.dmipy import DmipyMultiTissueModel

sphere = sphere_models.S4SphereGaussianPhaseApproximation(diffusion_constant=2.0e-9, diameter=12e-6)
ball = gaussian_models.G1Ball(lambda_iso=2.0e-9)
stick = cylinder_models.C1Stick(mu=[np.pi / 2, np.pi / 2], lambda_par=9.0e-9)

verdict_model = MultiCompartmentModel(models=[sphere, ball, stick])
# In Bonet-Carne et. al. the diameter of the intracellular sphere (d_IC) and the vascular diffusivity (P)
# can be fixed such that we can estimate d_{EES}#
#verdict_model.set_fixed_parameter('S4SphereGaussianPhaseApproximation_1_diameter', 10e-6)
verdict_model.set_fixed_parameter('C1Stick_1_lambda_par', 9.0e-9)

verdict_model.set_fixed_parameter('C1Stick_1_mu', [np.pi / 2, np.pi / 2])
verdict_model.set_fixed_parameter(
    'G1Ball_1_lambda_iso', 2.0e-9
)
# Ensure the sum hits exactly 1.0 in float64 representation
vfs = [0.6, 0.35]
vfs.append(1.0 - sum(vfs))

verdict_model = DmipyMultiTissueModel(verdict_model, vfs)


verdict_model["G1Ball_1_lambda_iso"].fit_flag = False
verdict_model["C1Stick_1_lambda_par"].fit_flag = False
relaxed_verdict_model = RelaxationTissueModel(model=verdict_model, t2=0.1)




relaxed_verdict_model["T2"].optimize = False
relaxed_verdict_model["T2"].fit_flag = False
relaxed_verdict_model["model_partial_volume_2"].optimize = False
print(relaxed_verdict_model)

/home/jgroot34/miniforge3/envs/MICROtool_2/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tissue model with 10 scalar parameters:
Tissue-parameter                                       Value    Scale  Optimize    Fit    Fit Bounds
---------------------------------------------------  -------  -------  ----------  -----  ------------------------------------------------
model_S4SphereGaussianPhaseApproximation_1_diameter  1.2e-05    1e-06  True        True   (1e-08, 1.9999999999999998e-05)
model_G1Ball_1_lambda_iso                            2e-09      1e-09  False       False  (1.0000000000000002e-10, 3.0000000000000004e-09)
model_C1Stick_1_mu_0                                 1.5708     1      False       True   (0.0, 3.141592653589793)
model_C1Stick_1_mu_1                                 1.5708     1      False       True   (-3.141592653589793, 3.141592653589793)
model_C1Stick_1_lambda_par                           9e-09      1e-09  False       False  (1.0000000000000002e-10, 3.0000000000000004e-09)
model_partial_volume_0                               0.6        1      True

In [2]:
import numpy as np
from microtool.acquisition_scheme import DiffusionAcquisitionScheme
from microtool.gradient_sampling import sample_uniform_half_sphere

n_directions = 6
n_shells = 2
block_size = n_directions + 1  # b0 + n_directions

shell_bvals = [1500, 2500]
shell_deltas = [0.0258, 0.0205]
shell_Deltas = [0.0434, 0.0331]
initial_te = 0.09

# Build the measurements in shell blocks: b0, then six directions.
b_vals_list = []
pulse_durations_list = []
pulse_intervals_list = []
echo_times_list = []

for i in range(n_shells):
    b_vals_list.append(0.0)
    b_vals_list.extend([shell_bvals[i]] * n_directions)

    pulse_durations_list.extend([shell_deltas[i]] * block_size)
    pulse_intervals_list.extend([shell_Deltas[i]] * block_size)
    echo_times_list.extend([initial_te] * block_size)

b_vals = np.array(b_vals_list)
pulse_durations = np.array(pulse_durations_list)
pulse_intervals = np.array(pulse_intervals_list)
echo_times = np.array(echo_times_list)
directions = sample_uniform_half_sphere(len(b_vals))

initial_scheme = DiffusionAcquisitionScheme.from_bvals(
    b_values=b_vals,
    b_vectors=directions,
    pulse_durations=pulse_durations,
    pulse_intervals=pulse_intervals,
    echo_times=echo_times,
)

# TE and timing: one free value for each 7-measurement shell block.
for name in (
    "EchoTime",
    "DiffusionPulseDuration",
    "DiffusionPulseInterval",
):
    initial_scheme[name].set_repetition_period(block_size)

# Gradient magnitude: fix b0, then share one value across each shell's six DW measurements.
b0_mask = b_vals == 0
magnitude = initial_scheme["DiffusionPulseMagnitude"]
magnitude.set_fixed_mask(b0_mask)
magnitude.set_repetition_period(n_directions)

print("TE free indices:",
      np.flatnonzero(initial_scheme["EchoTime"].optimize_mask))
print("Timing free indices:",
      np.flatnonzero(initial_scheme["DiffusionPulseDuration"].optimize_mask))
print("Gradient free indices:",
      np.flatnonzero(magnitude.optimize_mask))

print(initial_scheme)


TE free indices: [0 7]
Timing free indices: [0 7]
Gradient free indices: [1 8]
Acquisition scheme with 14 measurements and 6 scalar parameters. 
total number of optimized parameters is 8:
  DiffusionPulseMagnitude [mT/mm] in (0.0, 0.065)    DiffusionGradientAnglePhi [rad] (fixed)    DiffusionGradientAngleTheta [rad] (fixed)    DiffusionPulseDuration [s] in (0.001, 0.1)    DiffusionPulseInterval [s] in (0.001, 0.1)    EchoTime [s] in (0.001, 0.1)
-------------------------------------------------  -----------------------------------------  -------------------------------------------  --------------------------------------------  --------------------------------------------  ------------------------------
                                       -0                                          2.45176                                       1.5708                                          0.0258                                        0.0434                            0.09
                          

In [3]:
signal = relaxed_verdict_model(initial_scheme)
print(signal)

fitted_model = relaxed_verdict_model.fit(initial_scheme, signal, use_parallel_processing=False)
print(fitted_model.fitted_parameters)

jacobian = relaxed_verdict_model.scaled_jacobian(initial_scheme)
print(jacobian)

[0.40656966 0.23200724 0.25079329 0.2474323  0.23168768 0.23160212
 0.23160223 0.40656966 0.19702815 0.19703673 0.21039059 0.19705799
 0.19703691 0.21548705]
Setup brute2fine optimizer in 0.017500877380371094 seconds
Fitting of 1 voxels complete in 0.21686720848083496 seconds.
Average of 0.21686720848083496 seconds per voxel.
{'S4SphereGaussianPhaseApproximation_1_diameter': array([1.19995762e-05]), 'partial_volume_0': array([0.59998744]), 'partial_volume_1': array([0.35002816]), 'partial_volume_2': array([0.0499844])}
[[ 0.          0.          0.        ]
 [-0.0056004   0.36609195  0.01213822]
 [-0.0056004  -0.00962903 -0.36358276]
 [-0.0056004   0.05759071 -0.29636302]
 [-0.0056004   0.37248313  0.0185294 ]
 [-0.0056004   0.37419435  0.02024062]
 [-0.0056004   0.37419213  0.0202384 ]
 [ 0.          0.          0.        ]
 [-0.01236475  0.32678224  0.00273944]
 [-0.01236475  0.32661058  0.00256779]
 [-0.01236475  0.05953339 -0.2645094 ]
 [-0.01236475  0.32618533  0.00214254]
 [-0.01

In [4]:
from microtool.optimize import optimize_scheme

optimal_scheme, _ = optimize_scheme(initial_scheme, relaxed_verdict_model, noise_variance=0.05, method="trust-constr", solver_options = {
    "verbose": 2,
    "maxiter": 50,
    "finite_diff_rel_step": 1e-8,  # Prevents large jumps during gradient estimation
    "initial_tr_radius": 0.05        # Prevents trust radius from jumping to 35.0
})

/smb/user/jgroot34/DS-Data/Radiotherapie/Research/User/jgroot34/D_Projects/MICROtool/MICROtool_Github/microtool/microtool/tissue_model.py:533: UserWarning: If T2 relaxation is not used for fitting, echo time should be fixed.
  warnings.warn("If T2 relaxation is not used for fitting, echo time should be fixed.")


| niter |f evals|CG iter|  obj func   |tr radius |   opt    |  c viol  |
|-------|-------|-------|-------------|----------|----------|----------|
|   1   |   9   |   0   | +1.5358e+03 | 5.00e-02 | 9.08e+02 | 0.00e+00 |


/home/jgroot34/miniforge3/envs/MICROtool_2/lib/python3.8/site-packages/scipy/optimize/_hessian_update_strategy.py:182: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  warn('delta_grad == 0.0. Check if the approximated '


|   2   |  18   |   1   | +1.3901e+03 | 3.50e-01 | 7.74e+02 | 0.00e+00 |
|   3   |  27   |   8   | +1.0169e+03 | 1.45e+00 | 4.59e+02 | 0.00e+00 |
|   4   |  36   |  15   | +8.6780e+02 | 1.45e+00 | 3.42e+02 | 0.00e+00 |
|   5   |  45   |  22   | +7.9124e+02 | 1.45e+00 | 2.76e+02 | 0.00e+00 |
|   6   |  54   |  29   | +7.6771e+02 | 1.45e+00 | 2.50e+02 | 0.00e+00 |
|   7   |  63   |  36   | +7.4230e+02 | 1.45e+00 | 2.26e+02 | 0.00e+00 |
|   8   |  72   |  43   | +6.6340e+02 | 2.23e+00 | 1.67e+02 | 0.00e+00 |
|   9   |  81   |  50   | +5.2830e+02 | 4.15e+00 | 9.39e+01 | 0.00e+00 |
|  10   |  90   |  57   | +3.9011e+02 | 6.40e+00 | 5.01e+01 | 0.00e+00 |
|  11   |  99   |  64   | +3.1706e+02 | 8.00e+00 | 3.63e+01 | 0.00e+00 |
|  12   |  108  |  70   | +3.0626e+02 | 8.00e+00 | 2.90e+01 | 0.00e+00 |
|  13   |  117  |  76   | +2.8908e+02 | 8.00e+00 | 2.17e+01 | 0.00e+00 |
|  14   |  126  |  82   | +2.8578e+02 | 8.00e+00 | 2.14e+01 | 0.00e+00 |
|  15   |  135  |  88   | +2.8534e+02 | 8.00e+00 | 

/smb/user/jgroot34/DS-Data/Radiotherapie/Research/User/jgroot34/D_Projects/MICROtool/MICROtool_Github/microtool/microtool/optimize/optimize.py:246: UserWarning: Optimization procedure was unsuccessful. Possible solutions include but are not limited to: Changing the optimizer settings, changing the optimization method or changing the initial scheme to a more suitable one. If you are using a scipy optimizer its settings can be changed by passing options to this function. If you are using a MICROtool optimization method please consult the optimization_methods module for more details.
  warnings.warn(


In [5]:
print(optimal_scheme)
print(optimal_scheme.b_values)

Acquisition scheme with 14 measurements and 6 scalar parameters. 
total number of optimized parameters is 8:
  DiffusionPulseMagnitude [mT/mm] in (0.0, 0.065)    DiffusionGradientAnglePhi [rad] (fixed)    DiffusionGradientAngleTheta [rad] (fixed)    DiffusionPulseDuration [s] in (0.001, 0.1)    DiffusionPulseInterval [s] in (0.001, 0.1)    EchoTime [s] in (0.001, 0.1)
-------------------------------------------------  -----------------------------------------  -------------------------------------------  --------------------------------------------  --------------------------------------------  ------------------------------
                                       -0                                          2.45176                                       1.5708                                       0.0209623                                     0.0486929                       0.0828158
                                        0.0269863                                 -2.57288               